In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 08 · Registro del modelo

Se registra el modelo final en MLflow con su firma, parámetros, métricas de la prueba y el alias **`champion`**.
`configure_mlflow()` usa **Databricks** si `.env` tiene credenciales (Unity Catalog: `catalogo.esquema.libranzaradar_mora30`)
y, si no hay conexión, **MLflow local** (`mlflow_local.db`) como respaldo.

In [2]:
import json
import joblib
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
os.environ.setdefault("MLFLOW_ALLOW_PICKLE_DESERIALIZATION", "true")   # modelo propio, entrenado en el notebook 07
from proyecto.config import REPORTS_DIR, TARGET, GROUP_COL
from proyecto.models.train import split_xy
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

donde = configure_mlflow()
experimento = set_experiment("libranzaradar-08-registro")
modelo = joblib.load(ROOT / "models" / "final_pipeline.joblib")
resultados = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
dec = resultados["decisiones"]
train = joblib.load(ROOT / "data" / "processed" / "train.joblib")
cols = json.loads((REPORTS_DIR / "04_features_finales.json").read_text())["columnas"]
ejemplo = train[cols].head(5)
print("MLflow:", donde, "· experimento:", experimento)

2026/10/31 11:40:10 INFO mlflow.tracking.fluent: Experiment with name 'libranzaradar-08-registro' does not exist. Creating a new experiment.


MLflow: local · experimento: libranzaradar-08-registro


In [3]:
NOMBRE = registered_model_name("libranzaradar_mora30")
firma = infer_signature(ejemplo, modelo.predict_proba(ejemplo))
# Cuántas veces se abrió la prueba, leído del registro de aperturas (antes estaba escrito fijo "una vez": corregido 3-nov-2026)
aperturas = len([l for l in (REPORTS_DIR / "prueba_abierta.txt").read_text(encoding="utf-8").splitlines() if l.strip()])
with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params({"modelo": dec["modelo_final"], "origen_parametros": dec["origen_parametros"], "calibrado": dec["calibrar"],
                       "umbral": round(dec["umbral"], 6), **{f"p_{k}": v for k, v in dec["parametros"].items()}})
    mlflow.log_metrics({"test_recall": resultados["prueba"]["recall de mora (métrica principal)"]["modelo"][0],
                        "test_precision": resultados["prueba"]["precisión (exigida ≥ 20 %)"]["modelo"][0],
                        "test_pr_auc": resultados["prueba"]["PR-AUC"]["modelo"][0],
                        "test_auc": resultados["prueba"]["AUC"]["modelo"][0],
                        "test_flag_rate": resultados["porcentaje_marcado"]["modelo"]})
    mlflow.set_tags({"etapa": "08_registro", "caso": "libranzaradar", "objetivo": TARGET,
                     "particion": f"por cliente ({GROUP_COL})", "prueba_abierta": f"{aperturas} {'vez' if aperturas == 1 else 'veces'}"})
    mlflow.log_dict(dec, "decisiones_congeladas.json")
    info = mlflow.sklearn.log_model(modelo, name="modelo", signature=firma, input_example=ejemplo,
                                    serialization_format="cloudpickle")
    version = mlflow.register_model(info.model_uri, NOMBRE)
client = MlflowClient()
client.set_registered_model_alias(NOMBRE, "champion", version.version)
print(f"registrado {NOMBRE} versión {version.version} con alias champion (run {run.info.run_id})")

2026/10/31 11:40:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/31 11:40:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Successfully registered model 'libranzaradar_mora30'.


Created version '1' of model 'libranzaradar_mora30'.


registrado libranzaradar_mora30 versión 1 con alias champion (run 309f85c4e934454f838ea98d449953f4)


## Ida y vuelta: el modelo recuperado del registro predice igual que el local

In [4]:
recuperado = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
test = joblib.load(ROOT / "data" / "processed" / "test.joblib")
filas = test[cols].head(500)
dif = np.abs(recuperado.predict_proba(filas)[:, 1] - modelo.predict_proba(filas)[:, 1]).max()
print(f"diferencia máxima en 500 filas: {dif:.2e}")
assert dif < 1e-9
registro = {"plataforma": donde, "experimento": experimento, "modelo_registrado": NOMBRE, "version": int(version.version),
            "alias": "champion", "run_id": run.info.run_id, "model_uri": info.model_uri, "columnas_entrada": cols,
            "ida_y_vuelta_max_dif": float(dif), "mlflow": mlflow.__version__}
(REPORTS_DIR / "08_registro.json").write_text(json.dumps(registro, indent=2, ensure_ascii=False), encoding="utf-8")
registro

diferencia máxima en 500 filas: 0.00e+00


{'plataforma': 'local',
 'experimento': 'libranzaradar-08-registro',
 'modelo_registrado': 'libranzaradar_mora30',
 'version': 1,
 'alias': 'champion',
 'run_id': '309f85c4e934454f838ea98d449953f4',
 'model_uri': 'models:/m-0f2fb12735e84727a3fff8a189b9e719',
 'columnas_entrada': ['term_months',
  'loan_amount',
  'monthly_installment',
  'interest_rate_ea',
  'due_day',
  'has_guarantee',
  'guarantee_value',
  'has_cosigner',
  'product',
  'business_type',
  'economic_destination',
  'city'],
 'ida_y_vuelta_max_dif': 0.0,
 'mlflow': '3.17.1'}